# Demo. Decoder Basics

In [1]:
import stim
import pymatching
import sys
sys.path.append("../QEC-Codes")  # Adjust the path to import local modules
import numpy as np
from itertools import product
import matplotlib.pyplot as plt
from numpy.typing import NDArray
import pandas as pd
import seaborn as sns
import time
from sklearn.metrics import r2_score

This demo introduces the basics of decoders, focusing on pymatching and BP+OSD.  

**What does decoder do?**  
The input is the error information/syndromes/detection events, along with the detector error model that assigns probability/weights to each error mechanism. For each sample, the decoder will return a vector that gives the predictions for the logical errors that have happened (for surface code it's just one value because there is only one logical operator, for general codes it should a vector).

### 1. Pymatching for repetition codes

### Overall Analysis  
**LER decreases with increasing code distance and decreasing physical error rate (PER).**  
This behavior is consistent with theoretical expectations: as the code distance increases, more physical errors are required to cause a logical error. Similarly, reducing the physical error rate naturally reduces the probability of uncorrectable faults.

<!-- (2) **The break-even PER appears to be ≈ 1, meaning the repetition code always outperforms a single bit.**  
At first glance, this seems to contradict the classical result that repetition codes only outperform a single bit when \( p < 0.5 \) — since majority vote fails above this threshold. However, the classical result is based on **majority vote decoding**, whereas our experiment uses a **matching decoder** (e.g., `pymatching`), which finds the most likely error consistent with the observed syndrome, and can operate effectively even at high error rates. As a result, it can outperform majority vote and maintain logical error suppression for a much wider range of \( p \), even approaching 1.

(3) **The error threshold appears to be >0.5, meaning that increasing the code distance can suppresses the LER even when the PER is large.**  
In standard fault-tolerance theory, a threshold exists below which increasing the code distance improves logical fidelity, and above which larger codes perform worse. Here, because we are using the powerful **matching decoder** on this simple repetition code, the threshold become very large (>0.5). -->

### Analysis of LER Scaling

In principle, a QEC code of distance $d$ can correct up to $t = \lfloor \frac{d-1}{2} \rfloor$ errors. In the **ideal** case where all these errors are corrected, the leading-order contribution to the LER should be $p^{t+1}$, where $t + 1 = \lceil \frac{d+1}{2} \rceil$ is the weight of uncorrectable errors. In this case, a log-log plot of LER vs $p$ should produce a straight line with slope $t + 1$.

In realistic scenarios, however, decoding is imperfect: some correctable errors of weight ≤ $t$ may still lead to logical failure due to decoder suboptimality. As a result, the effective scaling exponent becomes **less than** $t + 1$.

Therefore, by fitting the log-log curve $\log(\text{LER})$ vs $ \log(p) $, we can estimate the effective suppression exponent. The **closer the slope is to t+1**, the more effective the decoder is at approaching the ideal case. Conversely, the difference between the slope and the ideal case quantifies suboptimal correction performance under the given noise model and decoding strategy. We can see that when d increases, the leading degrees become closer to t than t+1 and haves the tendency to drop even below t, indicating the fact that it becomes more and more difficult to correct all the errors up to weight t.  

The equation $$slope = (d_{eff}+1)/2 $$can give a metric called the "effective distance $d_{eff}$", which should be strictly smaller than the true distance $d$.

Next, we verify the "exponential suppression" of LER in code distance.

In [2]:
import sinter
from typing import List

This time we try to add the 2Q gate error and measurement error as well.

(1) We kind of see a "threshold behavior": the LER only suppresses as distance increases when PER is below 10.8\%. See where the d = 7 and d = 9 line intersect.  
(2) We also see a "break even" behavior: the LER is belowe PER when p < 10\%, where encoding into repetition code improves the error resilience compared to a bare qubit.

### 2. Pymatching for Surface Codes  
We directly use Sinter to streamline the sampling. This time we include the BPOSD decoder as well.

In [3]:
from bposdd import BPOSD
import BBcode
from stimbposd import SinterDecoder_BPOSD, sinter_decoders

In [4]:
tasks = []
p_list_extend = [1e-3,2e-3,3e-3,4e-3,5e-3]

d = 12

bb_code = BBcode.BBcode(n=144, k=12, d=12, m=6, l=12,
                        A=[[3, 0], [0, 1], [0, 2]],
                        B=[[0, 3], [1, 0], [2, 0]],
                        shift=[0, 0],
                        f=[[0, 0], [1, 0], [2, 0], [3, 0], [6, 0], [7, 0], [8, 0], [9, 0], [1, 3], [5, 3], [7, 3], [11, 3]],
                        g=[[1, 0], [2, 1], [0, 2], [1, 2], [2, 3], [0, 4]],
                        h=[[0, 0], [0, 1], [1, 1], [0, 2], [0, 3], [1, 3]],
                        alpha=[[0, 0], [0, 1], [2, 1], [2, 5], [3, 2], [4, 0]],
                        beta =[[0, 1], [0, 5], [1, 1], [0, 0], [4, 0], [5, 2]])
rounds = d
for p in p_list_extend:
    noise_profile = [p, p, p, p]
    bb_circuit = bb_code.build_full_BBcode_circuit(rounds=rounds, noise_profile = noise_profile, observable_type = "Z")
    tasks.append(sinter.Task(circuit=bb_circuit, json_metadata={'d': d, 'p1': noise_profile[0], 'p2': noise_profile[1], 'pM': noise_profile[2], 'pR': noise_profile[3]}))


collected_stats: List[sinter.TaskStats] = sinter.collect(
    num_workers=8,
    tasks=tasks,
    decoders=['bposd'],
    max_shots=1000, # change to large number for full scale experiment
    max_errors=100, # Whichever first hits the threshold stops the simulation
    custom_decoders = {
        'bposd': BPOSD(
            max_iter=1_000,
            bp_method = "ms",
            osd_order=10,
            osd_method="osd_cs"
        )
    },
    print_progress=True,
)


Starting 8 workers...
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
dem = bb_circuit.detector_error_model()
from beliefmatching import detector_error_model_to_check_matrices

pcm = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
print(pcm)


DemMatrices(check_matrix=<Compressed Sparse Column sparse matrix of dtype 'uint8'
	with 391320 stored elements and shape (1728, 67752)>, observables_matrix=<Compressed Sparse Column sparse matrix of dtype 'uint8'
	with 54108 stored elements and shape (12, 67752)>, edge_check_matrix=<Compressed Sparse Column sparse matrix of dtype 'uint8'
	with 3888 stored elements and shape (1728, 2016)>, edge_observables_matrix=<Compressed Sparse Column sparse matrix of dtype 'uint8'
	with 0 stored elements and shape (12, 2016)>, hyperedge_to_edge_matrix=<Compressed Sparse Column sparse matrix of dtype 'uint8'
	with 2016 stored elements and shape (2016, 67752)>, priors=array([0.0109224 , 0.00266667, 0.00100167, ..., 0.00632176, 0.00632176,
       0.00632176], shape=(67752,)))


In [ ]:
# This formula cannot capture large LER
# def covert_to_per_round(total_LER: float, rounds: int) -> float:
#     return 1 - (1 - total_LER)**(1/rounds)

# We should use this one based on the sum of boolean variables
def convert_to_per_round(total_LER: float, rounds: int) -> float:
    per_round_LER = 1/2 * (1 - np.abs(1-2*total_LER)**(1/rounds))
    return per_round_LER
# This will map total LER of 1/2 to per round LER of 1/2, indicating the purely unreliable estimation
# However, even with this more reasonable formulation, the estimation of per round LER is still not accurate when total_LER is around 1/2
# Only when total_LER is small can this per round conversion make perfect sense
    

def eval_LER(circuit: stim.Circuit, samples: NDArray[np.bool_], decoder = 'pymatching') -> float:
    detection_events = samples[:,:-1]
    observable_flips = samples[:,-1:]

    # Configure a decoder using the circuit.
    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    # Run the decoder.
    predictions = matcher.decode_batch(detection_events)

    # Count the mistakes.
    num_errors = np.sum(np.any(predictions != observable_flips, axis=1)) # this works for codes with multiple logical operators
    return num_errors/len(samples)

In [ ]:
# Convert collected_stats into Dataframe
records_sf = []
for stat in collected_stats:
    d = stat.json_metadata['d']
    rounds = d
    record = {
        'decoder': stat.decoder,
        'shots': stat.shots,
        'errors': stat.errors,
        'seconds': stat.seconds,
        'LER': convert_to_per_round(stat.errors / stat.shots, rounds) if stat.shots > 0 else None,
        'distance': d,
        'p1': stat.json_metadata['p1'], 'p2': stat.json_metadata['p2'], 'pM': stat.json_metadata['pM'], 'pR': stat.json_metadata['pR']
    }
    records_sf.append(record)
df_sf = pd.DataFrame(records_sf)
# df_sf.to_csv("results.csv", index=False) # If you wanna save the experiment data to csv
df_sf

,decoder,shots,errors,seconds,LER,distance,p1,p2,pM,pR
0,bposd,1000,0,1605.203719,0.000000,12,0.001,0.001,0.001,0.001
1,bposd,1000,0,4808.466772,0.000000,12,0.002,0.002,0.002,0.002
2,bposd,1000,5,9377.977817,0.000419,12,0.003,0.003,0.003,0.003
3,bposd,279,101,7246.368669,0.050865,12,0.005,0.005,0.005,0.005
4,bposd,1000,60,16406.652851,0.005298,12,0.004,0.004,0.004,0.004


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

# Figure 1: LER of Surf Codes
sns.lineplot(
    data=df_sf[df_sf.decoder == "bposd"],
    x="p2", y="LER",
    hue="distance",
    linewidth=2.5, alpha=0.75,
    palette=palette_distance, marker= 'o'
)
plt.xscale("log")
plt.yscale("log") # if you want a log-log plot
plt.title("Figure: LER of Surface Codes (Circuit-Level)")
plt.xlabel("Physical Error Rate (p)")
plt.ylabel("Logical Error Rate (LER)")
plt.grid(True)

# Draw the break-even lines
# x_vals = np.array(p_list)
# y_vals = x_vals
# plt.plot(x_vals, y_vals, color='red', linestyle='--', label='LER = PER')
# plt.legend()
# for d in d_list:
#     x_vals = np.array(p_list)
#     y_vals = 1 - (1 - x_vals * 2/3) ** d
#     plt.plot(x_vals, y_vals, color='red', linestyle='--')
#     plt.legend()

plt.tight_layout()
plt.show()

NameError: name 'palette_distance' is not defined

The analysis will be done in a seperate notebook because I use the server to run the simulation and output the result to a separate csv file.